# 2. Quem está sendo atendido (Figuras 6 e 7)

Reproduz a seção **"Quem está sendo atendido"** do site: as pirâmides de sexo e idade de 2025.

Em cada gráfico, as barras de mulheres e homens somam 100% e mostram como os atendidos por jogo patológico se
dividem por sexo e idade. Os traços mostram a mesma divisão na saúde mental em geral:
- **APS:** atendimentos com depressão, ansiedade, estresse, álcool ou esquizofrenia (recorte "sexo × faixa" do SISAB).
- **CAPS:** todos os pacientes com diagnóstico do capítulo F da CID-10, contados em pacientes-mês.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # para importar src/ (o notebook roda de dentro de notebooks/)

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt

from src.config import *
from src.util import *

In [ ]:
ANO = 2025

# APS: atendimentos por sexo × faixa
aps = pl.read_parquet(RAW / "atencao_primaria" / "sisab_aps.parquet")
aps = aps.filter((pl.col("recorte") == "sexofaixa") & pl.col("cmp").str.starts_with(str(ANO)))
aps = aps.with_columns(
    g=pl.when(pl.col("grupo").is_in(APS_JOGO)).then(pl.lit("jogo")).when(pl.col("grupo").is_in(APS_REF)).then(pl.lit("ref")),
    sexo=pl.col("valor").str.split("_").list.get(0), faixa=pl.col("valor").str.split("_").list.get(1))
aps = aps.drop_nulls("g").group_by("g", "sexo", "faixa").agg(n=pl.col("atendimentos").sum())

# CAPS: pacientes-mês com F63.0 e pacientes-mês com qualquer diagnóstico F
reg = pl.read_parquet(RAW / "caps" / "sia_f63_registros.parquet").filter(pl.col("cmp").str.starts_with(str(ANO)))
reg = reg.with_columns(cid=pl.when(pl.col("CIDPRI").str.starts_with("F63")).then(pl.col("CIDPRI")).otherwise(pl.col("CIDASSOC")))
caps_jogo = (reg.filter((pl.col("cid") == "F630") & pl.col("faixa").is_not_null()).unique(["id", "cmp"])
             .group_by(sexo="SEXOPAC", faixa="faixa").agg(n=pl.len().cast(pl.Int64)).with_columns(g=pl.lit("jogo")))
perfil = pl.read_parquet(RAW / "caps" / "sia_perfil.parquet")
caps_ref = (perfil.filter((pl.col("grupo") == "mental") & pl.col("cmp").str.starts_with(str(ANO))
                          & pl.col("SEXOPAC").is_in(["F", "M"]) & pl.col("faixa").is_not_null())
            .group_by(sexo="SEXOPAC", faixa="faixa").agg(n=pl.col("pacientes").sum().cast(pl.Int64)).with_columns(g=pl.lit("ref")))
caps = pl.concat([caps_jogo.select("g", "sexo", "faixa", "n"), caps_ref.select("g", "sexo", "faixa", "n")])

def percentuais(df):
    """% de cada sexo × faixa dentro do grupo (jogo ou referência); contagens < 5 ficam vazias."""
    df = df.with_columns(pct=pl.col("n") / pl.col("n").sum().over("g") * 100)
    return df.with_columns(pct=pl.when(pl.col("n") < MIN).then(None).otherwise(pl.col("pct")))

aps, caps = percentuais(aps), percentuais(caps)

In [ ]:
def piramide(df, cor, nome):
    """Barras: jogo patológico (mulheres à esquerda, homens à direita). Traços: saúde mental em geral."""
    fig, ax = plt.subplots(figsize=(12 * CM, 7 * CM))
    for i, faixa in enumerate(FAIXAS):
        for sexo, lado in (("F", -1), ("M", 1)):
            def v(g):
                x = df.filter((pl.col("g") == g) & (pl.col("sexo") == sexo) & (pl.col("faixa") == faixa))["pct"]
                return x[0] if len(x) and x[0] is not None else 0
            ax.barh(i, lado * v("jogo"), color=cor, height=0.7)
            ax.plot([lado * v("ref")] * 2, [i - 0.4, i + 0.4], color="black", lw=1.2)
    ax.axvline(0, color="black", lw=0.6)
    ax.set_yticks(range(len(FAIXAS)), [f.replace("-", "–") for f in FAIXAS])
    ax.set_ylabel("Faixa etária (anos)")
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: br(abs(v), 0)))
    ax.set_xlabel("% dos atendidos")
    lim = max(abs(x) for x in ax.get_xlim())
    ax.set_xlim(-lim, lim)
    ax.text(0.25, 1.02, "Mulheres", transform=ax.transAxes, ha="center")
    ax.text(0.75, 1.02, "Homens", transform=ax.transAxes, ha="center")
    salvar_figura(fig, nome)

from matplotlib.ticker import FuncFormatter
piramide(aps, COR["aps"], "figura6")
piramide(caps, COR["caps"], "figura7")

## Números citados no texto

In [ ]:
def parte(df, g, sexos=("F", "M"), faixas=FAIXAS):
    x = df.filter((pl.col("g") == g) & pl.col("sexo").is_in(sexos) & pl.col("faixa").is_in(faixas))
    return 100 * x["n"].sum() / df.filter(pl.col("g") == g)["n"].sum()

num = {}
for nome, df in (("aps", aps), ("caps", caps)):
    for g in ("jogo", "ref"):
        num[f"{nome}_{g}_25a44"] = br(parte(df, g, faixas=["25-34", "35-44"]), 0)
        num[f"{nome}_{g}_45mais"] = br(parte(df, g, faixas=["45-59", "60+"]), 0)
        num[f"{nome}_{g}_menores"] = br(parte(df, g, faixas=["0-17"]), 0)
        num[f"{nome}_{g}_mulheres"] = br(parte(df, g, sexos=["F"]), 0)
        num[f"{nome}_{g}_homens_18a24"] = br(parte(df, g, sexos=["M"], faixas=["18-24"]), 0)
num  # números citados no texto